In [3]:
import pandas as pd
from pandas_plink import read_plink1_bin, write_plink1_bin
import numpy as np
import anndata
import os

/cis/home/xhan56/anaconda3/envs/mmt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
adni_1_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI_1_GWAS_Plink/ADNI_cluster_01_forward_757LONI'
adni_2_1_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI_GO_2_OmniExpress/ADNI_GO_2_Forward_Bin'
adni_2_2_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI_GO2_2nd/Binary/ADNI_GO2_GWAS_2nd_orig_BIN'
adni_3_1_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI3_PLINKFinal/PLINK_Final/ADNI3_PLINK_Final'
adni_3_2_path = '/export/io79/data/schaud35/datasets/adni/genomic/PLINKFinal_ADNI_20231107/PLINK_Final/ADNI3_PLINK_FINAL_2nd'

# 1. Prepare liftover format

In [6]:
OUTPUT_ROOT = '/export/io79/data/schaud35/datasets/adni/processed/genomic'
BEFORE_LIFTOVER_DIR = f'{OUTPUT_ROOT}/before_liftover'
AFTER_LIFTOVER_DIR = f'{OUTPUT_ROOT}/after_liftover'
LIFTOVERED_DIR = f'{OUTPUT_ROOT}/liftovered'

os.makedirs(BEFORE_LIFTOVER_DIR, exist_ok=True)
os.makedirs(AFTER_LIFTOVER_DIR, exist_ok=True)
os.makedirs(LIFTOVERED_DIR, exist_ok=True)

PermissionError: [Errno 13] Permission denied: '/export'

In [ ]:
adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path] 

for i in range(5):
    df = pd.read_csv('./' + adni_paths[i] + '.bim', header=None, sep='\t')
    
    valid_bases = ['A', 'C', 'G', 'T']
    df_filtered = df[df[4].isin(valid_bases) & df[5].isin(valid_bases)]
    df_filtered = df_filtered.reset_index(drop=True)
    df_filtered.to_csv('./' + adni_paths[i] + '.bim', index=False, header=None, sep='\t')

In [ ]:
adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path]
save_paths = ['adni_1.bed', 'adni_2_1.bed', 'adni_2_2.bed', 'adni_3_1.bed', 'adni_3_2.bed']
# os.makedirs('./before_liftover', exist_ok=True)

for i in range(len(adni_paths)):
    bim_path = adni_paths[i]
    save_path = os.path.join(BEFORE_LIFTOVER_DIR, save_paths[i])
    tmp = pd.read_csv(f'./{bim_path}.bim', header=None, sep='\t')
    
    chr_id = tmp[0].replace({23: 'X', 24: 'Y'}).where((0 < tmp[0]) & (tmp[0] < 25), None)
    
    filtered_data = tmp[chr_id.notna()]
    
    new_tmp = pd.DataFrame({
        '0': 'chr' + chr_id[chr_id.notna()].astype(str),
        '1': filtered_data[3],
        '2': filtered_data[3],
        '3': filtered_data[1]
    })
    
    new_tmp = new_tmp.reset_index(drop=True)
    new_tmp.to_csv(save_path, index=False, sep='\t', header=None)

# 2. Run Liftover & Filtering

In [ ]:
### --------------- Liftover Process (Run following commandas at the terminal) --------------- ###
# wget http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/liftOver
# chmod +x liftOver
# wget http://hgdownload.soe.ucsc.edu/goldenPath/hg18/liftOver/hg18ToHg38.over.chain.gz
# ./liftOver ./before_liftover/adni_1.bed hg18ToHg38.over.chain.gz ./after_liftover/ADNI_1_Hg38.bed ./after_liftover/unlifted_ADNI_1_Hg38.bed 
# ./liftOver ./before_liftover/adni_2_1.bed hg18ToHg38.over.chain.gz ./after_liftover/ADNI_2_1_Hg38.bed ./after_liftover/unlifted_ADNI_2_1_Hg38.bed 
# ./liftOver ./before_liftover/adni_2_2.bed hg18ToHg38.over.chain.gz ./after_liftover/ADNI_2_2_Hg38.bed ./after_liftover/unlifted_ADNI_2_2_Hg38.bed 
# ./liftOver ./before_liftover/adni_3_1.bed hg18ToHg38.over.chain.gz ./after_liftover/ADNI_3_1_Hg38.bed ./after_liftover/unlifted_ADNI_3_1_Hg38.bed 
# ./liftOver ./before_liftover/adni_3_2.bed hg18ToHg38.over.chain.gz ./after_liftover/ADNI_3_2_Hg38.bed ./after_liftover/unlifted_ADNI_3_2_Hg38.bed 

In [ ]:
# filtering
# os.makedirs('./after_liftover', exist_ok=True)
# after_liftover_paths = ['after_liftover/ADNI_1', 'after_liftover/ADNI_2_1', 'after_liftover/ADNI_2_2', 'after_liftover/ADNI_3_1', 'after_liftover/ADNI_3_2']

# for after_liftover_path in after_liftover_paths:
#     _tmp = pd.read_csv(f'./' + after_liftover_path + '_Hg38.bed', header=None, sep='\t')
#     _tmp_filtered = _tmp[~_tmp[0].str.contains('_')]
#     _tmp_filtered.to_csv(f'./' + after_liftover_path + '_Hg38.bed', index=False, header=False, sep='\t')
after_liftover_bases = ['ADNI_1', 'ADNI_2_1', 'ADNI_2_2', 'ADNI_3_1', 'ADNI_3_2']
after_liftover_paths = [os.path.join(AFTER_LIFTOVER_DIR, b) for b in after_liftover_bases]
for after_liftover_path in after_liftover_paths:
    bed_path = after_liftover_path + '_Hg38.bed'
    _tmp = pd.read_csv(bed_path, header=None, sep='\t')
    _tmp_filtered = _tmp[~_tmp[0].str.contains('_')]
    _tmp_filtered.to_csv(bed_path, index=False, header=False, sep='\t')

# 3. Align .bim, .bed, .fam with liftover ids

In [ ]:
# adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path]
# liftover_paths = ['after_liftover/ADNI_1', 'after_liftover/ADNI_2_1', 'after_liftover/ADNI_2_2', 'after_liftover/ADNI_3_1', 'after_liftover/ADNI_3_2']
# output_paths = ['liftovered/ADNI_1', 'liftovered/ADNI_2_1', 'liftovered/ADNI_2_2', 'liftovered/ADNI_3_1', 'liftovered/ADNI_3_2']
# os.makedirs('./liftovered', exist_ok=True)

# for i in range(5):
#     adni_path = './' + adni_paths[i]
#     liftover_path = './' + liftover_paths[i] + '_Hg38.bed'
#     output_path = './' + output_paths[i]
    
#     G = read_plink1_bin(adni_path+'.bed', adni_path+'.bim', adni_path+'.fam')

#     # Read the liftover file to get the new SNP IDs
#     liftover = pd.read_csv(liftover_path, sep='\t', header=None, names=['chr', 'pos1', 'pos2', 'id'])

#     # Find indices of the SNPs in bim that are present in the liftover file
#     indices = np.arange(G.shape[1])[(pd.DataFrame(G.snp)[0].isin(liftover['id']))]

#     if i == 0:
#         fam = pd.read_csv('./ADNI_cluster_01_forward_757LONI.fam', sep=' ', header=None, names=['fid', 'iid', 'father', 'mother', 'gender', 'trait'])
#         fam_indices = ~fam['iid'].isin(['073_S_0909', '130_S_1201'])
#         _G = G[fam_indices, indices]

#     else:
#         _G = G[:, indices]
    
#     _G['chrom'] = ('variant', np.array(liftover['chr']))
#     _G['pos'] = ('variant', np.array(liftover['pos1']))
#     write_plink1_bin(_G, output_path+'.bed', output_path+'.bim', output_path+'.fam')

adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path]
after_liftover_bases = ['ADNI_1', 'ADNI_2_1', 'ADNI_2_2', 'ADNI_3_1', 'ADNI_3_2']
liftover_paths = [os.path.join(AFTER_LIFTOVER_DIR, b) for b in after_liftover_bases]
output_paths = [os.path.join(LIFTOVERED_DIR, b) for b in after_liftover_bases]
for i in range(5):
    adni_path = adni_paths[i]
    liftover_path = liftover_paths[i] + '_Hg38.bed'
    output_path = output_paths[i]
    G = read_plink1_bin(adni_path + '.bed', adni_path + '.bim', adni_path + '.fam')
    # Read the liftover file to get the new SNP IDs
    liftover = pd.read_csv(liftover_path, sep='\t', header=None, names=['chr', 'pos1', 'pos2', 'id'])
    # Find indices of SNPs present in liftover output
    indices = np.arange(G.shape[1])[(pd.DataFrame(G.snp)[0].isin(liftover['id']))]
    if i == 0:
        fam = pd.read_csv(adni_1_path + '.fam', sep=r'\s+', header=None,
                          names=['fid', 'iid', 'father', 'mother', 'gender', 'trait'])
        fam_indices = ~fam['iid'].isin(['073_S_0909', '130_S_1201'])
        _G = G[fam_indices, indices]
    else:
        _G = G[:, indices]
    _G['chrom'] = ('variant', np.array(liftover['chr']))
    _G['pos'] = ('variant', np.array(liftover['pos1']))
    write_plink1_bin(_G, output_path + '.bed', output_path + '.bim', output_path + '.fam')

In [ ]:
G_list = []
for i in range(5):
    adni_path = './' + output_paths[i]
    
    G = read_plink1_bin(adni_path+'.bed', adni_path+'.bim', adni_path+'.fam')
    G_list.append(G)

# 4.Merge using plink on your local enviroment

In [ ]:
### --------------- Merge using plink (Run following commandas at the terminal) --------------- ###
# wget https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20241022.zip
# unzip plink_linux_x86_64_20241022.zip -d plink_install
# cd plink_install
# cp plink /usr/local/bin
# sudo cp plink /usr/local/bin
# sudo chmod 755 /usr/local/bin/plink
# sudo nano ~/.bashrc -> export PATH=/usr/local/bin:$PATH
# cd ..
# echo "./liftovered/ADNI_2_1" > ./liftovered/all_datasets.txt
# echo "./liftovered/ADNI_2_2" >> ./liftovered/all_datasets.txt
# echo "./liftovered/ADNI_3_1" >> ./liftovered/all_datasets.txt
# echo "./liftovered/ADNI_3_2" >> ./liftovered/all_datasets.txt
# plink --bfile ./liftovered/ADNI_1 --merge-list ./liftovered/all_datasets.txt --make-bed --out ./liftovered/ADNI_merged

In [ ]:
### --------------- LD pruning using plink (Run following commandas at the terminal) --------------- ###
# plink --bfile ./liftovered/ADNI_merged --indep-pairwise 50 5 0.1 --out ./liftovered/ADNI_merged_pruned
# plink --bfile ./liftovered/ADNI_merged --extract ./liftovered/ADNI_merged_pruned.prune.in --make-bed --out ./liftovered/ADNI_final

In [ ]:
# merged_path = "./liftovered/ADNI_final"
merged_path = f"{LIFTOVERED_DIR}/ADNI_final"
G_merge = read_plink1_bin(merged_path+'.bed', merged_path+'.bim', merged_path+'.fam')

genotype_df = G_merge.to_pandas()
genotype_df.columns = G_merge.snp

adata = anndata.AnnData(X=genotype_df.values,
                        obs=pd.DataFrame(index=genotype_df.index),
                        var=pd.DataFrame(index=genotype_df.columns))

adata.write_h5ad("genomic_merged.h5ad")